# Sprint 5 — RBF SVM (kernel, on a subsample)

The non-linear SVM. The question: can a **curved** boundary climb back toward the tree, where the straight `LinearSVC` (0.9412 / 0.8875) could not?

A kernel SVM is ~O(n^2) in training rows, so the full 2.26M is infeasible. We train on a **stratified 50k subsample** (keeps the 19.70% malicious ratio), still the featured split (bucketed port), scaled train-only. No `class_weight` yet, baseline first (Section 7 #12).

Benchmarks: LinearSVC 0.9412 / 0.8875 · bucketed tree 0.9990 / 0.9984.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

proc = Path('../data/processed/featured')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')['label_binary']
y_test  = pd.read_parquet(proc / 'y_test.parquet')['label_binary']
print('featured split:', X_train.shape, '|', X_test.shape)

## 1. Subsample + scale

Kernel-SVM training scales ~quadratically, so we take a **stratified 50k** slice of train (19.70% malicious preserved). Scale with `StandardScaler` fit on that subsample only (train-only, Section 7 #10).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

N = 50_000
X_sub, _, y_sub, _ = train_test_split(X_train, y_train, train_size=N, stratify=y_train, random_state=42)
print('train subsample:', X_sub.shape, '| malicious %.4f' % y_sub.mean())

scaler = StandardScaler().fit(X_sub)           # fit on the subsample only
X_sub_s  = scaler.transform(X_sub)
X_test_s = scaler.transform(X_test)

## 2. Train the RBF SVM

`kernel='rbf'` bends the boundary; `gamma='scale'` and `C=1.0` are the hand-reasoned defaults (Section 8: no big hyperparameter search). No `class_weight` (baseline).

In [ ]:
from sklearn.svm import SVC
from time import perf_counter

t0 = perf_counter()
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_sub_s, y_sub)
print('fit in %.1fs | support vectors: %d of %d' % (perf_counter() - t0, svm.support_.size, N))

## 3. Evaluate (full test) and compare

Note: predicting the full 566k test with a kernel SVM takes ~1-2 min (each point is compared against every support vector). Benchmarks: LinearSVC 0.9412 / 0.8875, bucketed tree 0.9990 / 0.9984.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report
from time import perf_counter

t0 = perf_counter()
y_pred = svm.predict(X_test_s)
print('predict (full test) in %.1fs\n' % (perf_counter() - t0))

print('accuracy  : %.4f   (LinearSVC 0.9412 | tree 0.9990)' % accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print('\nconfusion matrix (rows = actual, cols = predicted):')
print(pd.DataFrame(cm, index=['actual benign', 'actual malicious'], columns=['pred benign', 'pred malicious']))
print('\nFN (missed intrusions): %d   |   FP (false alarms): %d' % (cm[1, 0], cm[0, 1]))
print('\n' + classification_report(y_test, y_pred, target_names=['benign', 'malicious'], digits=4))
print('recall (malicious): %.4f   (LinearSVC 0.8875 | tree 0.9984)' % recall_score(y_test, y_pred, zero_division=0))